# Notebook 6 — Does SNR explain which configuration wins?

**Purpose:** answer one question — do some configurations do better than the baseline on
*quiet* files, and others on *loud* ones?

**Inputs:** `snr/per_file_snr.csv` from `snr_analysis.ipynb`, and the run logs. No audio
needed, so this runs on any machine once the csv is committed.

**Outputs** (into `snr/`): `snr_by_config_bin.csv`, `snr_delta_vs_baseline.csv`,
`snr_corr_by_config.csv`, `snr_by_subset_config.csv`, and two figures.

Files are grouped two ways — **terciles** (equal file counts, so no bin is empty) and
**fixed dB edges** (comparable to any other paper) — and every table is produced for both.
If the two groupings disagree, the bin edges are doing the talking rather than the data.


## 0 — Configuration

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(".").resolve()))
import parse_logs as pl

RUN_LOGS = Path("../../../paper/run_logs").resolve()

# The canonical seven labels, in the order used by the other notebooks.
EXPERIMENT_LOGS = {
    "Baseline":                 RUN_LOGS / "baseline.log",
    "CIFE":                     RUN_LOGS / "CIFE.log",
    "Phase 1 (Random)":         RUN_LOGS / "phase_1.log",
    "Phase 2 (Energy)":         RUN_LOGS / "phase_2.log",
    "Phase 3 (Activation)":     RUN_LOGS / "phase_3.log",
    "Phase 3 + AFRI (Direct)":  RUN_LOGS / "afri_direct.log",
    "Phase 4 + AFRI (Inverse)": RUN_LOGS / "afri_inverse.log",
}
BASELINE_LABEL = "Baseline"

SNR_CSV = Path("snr/per_file_snr.csv")
OUT_DIR = Path("snr")
OUT_DIR.mkdir(exist_ok=True)

# Fixed dB edges for the absolute grouping (-inf and +inf are appended)
SNR_EDGES = [5.0, 15.0]

logs_to_parse = {k: str(v) for k, v in EXPERIMENT_LOGS.items() if v is not None and v.exists()}
for label, path in EXPERIMENT_LOGS.items():
    print(f"{'ok ' if label in logs_to_parse else 'MISSING'}  {label:<26} {path}")
print(f"\n{len(logs_to_parse)} of {len(EXPERIMENT_LOGS)} logs found")

## 1 — Load the results and the SNR table

In [ ]:
experiments = pl.load_experiments(logs_to_parse)

df_pf = pl.per_file_df(experiments)          # per-file TP/FP/FN at each run's winning threshold
df_pf["fmeasure"] *= 100                     # per_file_df is 0-1; everything here is 0-100
df_pf["precision"] *= 100
df_pf["recall"] *= 100

if not SNR_CSV.exists():
    raise FileNotFoundError(f"{SNR_CSV} not found -- run snr_analysis.ipynb first "
                            "(it needs the dataset mounted)")
df_snr = pd.read_csv(SNR_CSV)

print(f"per-file rows : {len(df_pf)}  ({df_pf['experiment'].nunique()} configs x "
      f"{df_pf['filename'].nunique()} files)")
print(f"snr rows      : {len(df_snr)}  ({', '.join(df_snr['split'].unique())})")
if "from_backfill" in df_pf and df_pf["from_backfill"].any():
    n_bf = df_pf.loc[df_pf["from_backfill"], "filename"].nunique()
    print(f"backfilled    : {n_bf} file(s) the logs never printed, counted as TP=0 "
          "(these are the total misses -- the interesting low-SNR cases)")
else:
    print("backfilled    : none -- if notebooks/file_to_subset.json is missing, files with "
          "no predictions\n                are absent entirely and every number below is "
          "optimistic. Run\n                tools/build_subset_map.py (or cell 6 of "
          "snr_analysis.ipynb) on the data machine.")

### Merge

`subset` is taken from the SNR csv, which derives it from the dataset folder and so
distinguishes `PB` from `PB24` — `parse_logs` can only infer it from the filename prefix
when `file_to_subset.json` is absent, and those two folders both hold `BUK*` files.

In [ ]:
SNR_COLS = ["filename", "subset", "split", "duration_s", "n_events",
            "event_dur_median_s", "snr_db", "snr_db_scored", "at_floor"]

# Everything downstream is keyed on the bare filename -- the logs have nothing else to go
# on -- so two files sharing a name would fan out into duplicate rows here and their
# TP/FP/FN would be counted twice in every pooled score. Keep the first by path.
snr_join = df_snr.sort_values("path")
dupes = sorted(snr_join.loc[snr_join.duplicated("filename", keep=False), "filename"].unique())
if dupes:
    warnings.warn(f"{len(dupes)} filename(s) appear more than once in {SNR_CSV}: {dupes}. "
                  "Keeping the first by path, since the logs cannot distinguish them.")
snr_join = snr_join.drop_duplicates("filename", keep="first")

df = df_pf.drop(columns=["subset"]).merge(snr_join[SNR_COLS], on="filename", how="inner")

# One row per (config, file) and no more. Configs are allowed to cover different files
# here -- that is the backfill question, warned about separately below -- but a file must
# never appear twice for the same config, or its counts get double-weighted.
n_pairs = len(df.drop_duplicates(["experiment", "filename"]))
if len(df) != n_pairs:
    raise RuntimeError(f"The merge produced {len(df)} rows for {n_pairs} distinct "
                       "(config, file) pairs, so some counts would be double-weighted. "
                       "Check for repeated filenames in the SNR csv.")

logged = set(df_pf["filename"])
validation = set(df_snr.loc[df_snr["split"] == "Validation", "filename"])
missing_snr = sorted(logged - set(df_snr["filename"]))
unlogged = sorted(validation - logged)

print(f"joined: {df['filename'].nunique()} files x {df['experiment'].nunique()} configs "
      f"= {len(df)} rows")

# Each config must be scored over the SAME files, or the bins are not comparable: a log
# only prints files that had at least one prediction, and the seven runs here printed
# 40-42 files each. parse_logs backfills the rest as total misses -- but only when
# file_to_subset.json exists.
per_config = df.groupby("experiment")["filename"].nunique()
if per_config.nunique() != 1:
    warnings.warn(
        "Configs cover different file sets, so every comparison below is biased: "
        f"{per_config.to_dict()}. Build notebooks/file_to_subset.json "
        "(tools/build_subset_map.py, or cell 6 of snr_analysis.ipynb) so the files with "
        "no predictions are backfilled as total misses."
    )
if missing_snr:
    print(f"\nIn the logs but not in the SNR csv ({len(missing_snr)}) -- the csv and the logs "
          "are from different dataset revisions:")
    for name in missing_snr:
        print(f"   {name}")
if unlogged:
    print(f"\nMeasured but never scored ({len(unlogged)}) -- expected only if the backfill "
          "is off:")
    for name in unlogged:
        print(f"   {name}")

no_snr = df.loc[df["snr_db"].isna(), "filename"].unique()
if len(no_snr):
    print(f"\nDropping {len(no_snr)} file(s) with no usable SNR: {', '.join(no_snr)}")
    df = df.dropna(subset=["snr_db"])

## 2 — Bin the files

Both groupings are added as columns, and every table below is built for each.

In [ ]:
TERCILE_LABELS = ["low", "mid", "high"]

per_file = df.drop_duplicates("filename").set_index("filename")["snr_db"]

tercile_edges = per_file.quantile([0, 1 / 3, 2 / 3, 1]).to_numpy()
tercile = pd.qcut(per_file, 3, labels=TERCILE_LABELS)

band_edges = [-np.inf] + SNR_EDGES + [np.inf]
band_labels = ([f"< {SNR_EDGES[0]:g} dB"]
               + [f"{lo:g}-{hi:g} dB" for lo, hi in zip(SNR_EDGES, SNR_EDGES[1:])]
               + [f"> {SNR_EDGES[-1]:g} dB"])
band = pd.cut(per_file, band_edges, labels=band_labels)

df["snr_tercile"] = df["filename"].map(tercile).astype(str)
df["snr_band"] = df["filename"].map(band).astype(str)

GROUPINGS = {"snr_tercile": TERCILE_LABELS, "snr_band": band_labels}

print("terciles (equal file counts):")
for lab, lo, hi in zip(TERCILE_LABELS, tercile_edges, tercile_edges[1:]):
    n = int((tercile == lab).sum())
    print(f"   {lab:<5} {lo:6.1f} to {hi:6.1f} dB   {n} files")
print("\nfixed edges:")
for lab in band_labels:
    files = per_file[band == lab]
    rng = f"{files.min():.1f} to {files.max():.1f} dB" if len(files) else "--"
    print(f"   {lab:<10} {len(files)} files   {rng}")

## 3 — Per-configuration scores per SNR bin

**Micro** pools TP/FP/FN across the files in a bin and scores the totals — the
evaluator's own aggregation, reused straight from `parse_logs.subset_scores_df`
(which pools on `(experiment, subset)`, so the bin column is passed in as `subset`).
**Macro** is the plain mean of the per-file F-measures. They disagree on this dataset:
micro is dominated by the files with many events, macro treats every file equally.

In [ ]:
def pooled_scores(frame, by, labels):
    """Evaluator-style pooled (micro) scores per (experiment, bin), plus macro mean.

    subset_scores_df pools on (experiment, subset), so the bin column is handed to it
    as `subset` -- the real subset column has to go first, or the rename produces two
    columns of that name and the groupby fails.
    """
    tmp = frame.drop(columns=["subset"], errors="ignore").rename(columns={by: "subset"})
    tmp["subset"] = tmp["subset"].astype(str)
    micro = pl.subset_scores_df(tmp).rename_axis(index=["experiment", by])

    macro = (frame.groupby(["experiment", by])
                  .agg(files=("filename", "nunique"),
                       macro_f=("fmeasure", "mean"),
                       snr_med=("snr_db", "median"))
                  .rename_axis(index=["experiment", by]))

    out = micro.join(macro).reset_index()
    out[by] = pd.Categorical(out[by], categories=labels, ordered=True)
    out = out.sort_values(["experiment", by]).set_index(["experiment", by])
    return out.rename(columns={"precision": "micro_p", "recall": "micro_r",
                               "fmeasure": "micro_f"})


tables = {by: pooled_scores(df, by, labels) for by, labels in GROUPINGS.items()}

by_bin = pd.concat(
    [t.rename_axis(index=["experiment", "bin"]).assign(grouping=by).reset_index()
     for by, t in tables.items()],
    ignore_index=True,
)[["grouping", "experiment", "bin", "files", "TP", "FP", "FN",
   "micro_p", "micro_r", "micro_f", "macro_f", "snr_med"]].round(2)
by_bin.to_csv(OUT_DIR / "snr_by_config_bin.csv", index=False)
print(f"saved {OUT_DIR / 'snr_by_config_bin.csv'}")

tables["snr_tercile"][["files", "TP", "FP", "FN", "micro_f", "macro_f", "snr_med"]].round(2)

### Sanity check — does the binning preserve the official score?

Pooling every bin back together must reproduce each run's reported F-measure.

- `diff ≈ 0` — the reconstruction is sound and nothing was lost in the merge.
- **`diff` positive for every config, by a few points** — the file set is incomplete, not
  the arithmetic: files the logs never printed are missing their FNs. Build
  `notebooks/file_to_subset.json` (cell 6 of `snr_analysis.ipynb`, or
  `tools/build_subset_map.py`) and re-run.
- `diff` differing in sign between configs — that one is a merge bug worth chasing.

In [ ]:
check = (pooled_scores(df.assign(all_files="all"), "all_files", ["all"])
         .reset_index(level=1, drop=True)[["micro_f"]]
         .join(pl.final_results_df(experiments)[["fmeasure"]])
         .rename(columns={"micro_f": "pooled_here", "fmeasure": "reported_in_log"}))
check["diff"] = (check["pooled_here"] - check["reported_in_log"]).round(3)
check.round(3)

## 4 — Delta vs baseline

The direct answer: for each configuration and each SNR bin, F-measure minus the
baseline's F-measure in that same bin. Positive means the configuration beats the
baseline on those files.

In [ ]:
def delta_grid(table, by, metric="micro_f"):
    grid = table.reset_index().pivot(index="experiment", columns=by, values=metric)
    grid = grid.reindex([k for k in EXPERIMENT_LOGS if k in grid.index])
    if BASELINE_LABEL not in grid.index:
        raise KeyError(f"{BASELINE_LABEL!r} is not among the loaded configs "
                       f"({list(grid.index)}) -- there is nothing to compare against")
    return (grid - grid.loc[BASELINE_LABEL]).drop(index=BASELINE_LABEL)


deltas = {}
for metric in ("micro_f", "macro_f"):
    for by, labels in GROUPINGS.items():
        deltas[(metric, by)] = delta_grid(tables[by], by, metric)

pd.concat(
    [g.assign(metric=metric, grouping=by).rename_axis(index="experiment").reset_index()
     for (metric, by), g in deltas.items()],
    ignore_index=True,
).round(2).to_csv(OUT_DIR / "snr_delta_vs_baseline.csv", index=False)
print(f"saved {OUT_DIR / 'snr_delta_vs_baseline.csv'}\n")
print("micro F-measure, delta vs Baseline, by tercile:")
deltas[("micro_f", "snr_tercile")].round(2)

## 5 — Correlation, and the files each configuration misses entirely

Two bin-free views, so the conclusion doesn't hinge on where the edges fall:

- **Spearman ρ** between per-file SNR and per-file F-measure (rank correlation, robust to
  the pile-up of zeros). Positive ρ = the configuration does better on louder files.
- **hit / miss SNR** — the median SNR of the files where the configuration found something
  (F > 0) versus the files it missed completely (F = 0). If the miss column sits well below
  the hit column, the failures really are the quiet files.

In [ ]:
from scipy.stats import spearmanr

rows = []
for label in [k for k in EXPERIMENT_LOGS if k in set(df["experiment"])]:
    g = df[df["experiment"] == label]
    row = {"experiment": label, "files": g["filename"].nunique()}
    for metric in ("fmeasure", "recall", "precision"):
        rho, p = spearmanr(g["snr_db"], g[metric])
        row[f"rho_{metric[:4]}"] = rho
        row[f"p_{metric[:4]}"] = p
    hit, miss = g[g["fmeasure"] > 0], g[g["fmeasure"] == 0]
    row["n_miss"] = len(miss)
    row["snr_med_hit"] = hit["snr_db"].median()
    row["snr_med_miss"] = miss["snr_db"].median()
    rows.append(row)

corr = pd.DataFrame(rows).set_index("experiment").round(3)
corr.to_csv(OUT_DIR / "snr_corr_by_config.csv")
print(f"saved {OUT_DIR / 'snr_corr_by_config.csv'}")
corr

### Per subset, with the subset's SNR alongside

This separates *"this configuration is better on quiet files"* from *"this configuration is
better on ME, which happens to be quiet"* — with ~6 validation subsets, the two are easy to
confuse.

In [ ]:
subset_f = (pl.subset_scores_df(df)["fmeasure"]
            .unstack("subset")
            .reindex([k for k in EXPERIMENT_LOGS if k in set(df["experiment"])]))
subset_snr = df.drop_duplicates("filename").groupby("subset")["snr_db"].median()

by_subset = pd.concat([subset_f, pd.DataFrame({"median SNR (dB)": subset_snr}).T])
by_subset = by_subset[subset_snr.sort_values().index]      # columns quietest -> loudest
by_subset.round(2).to_csv(OUT_DIR / "snr_by_subset_config.csv")
print(f"saved {OUT_DIR / 'snr_by_subset_config.csv'}   (columns sorted quietest -> loudest)")
by_subset.round(2)

## 6 — Figures

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.4,
    "font.size": 10,
})

# Fixed categorical order -- a configuration keeps its colour in every figure.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
CONFIG_COLOR = {label: PALETTE[i % len(PALETTE)] for i, label in enumerate(EXPERIMENT_LOGS)}
INK, GREY = "#3a3a3a", "#9a9a9a"

# Diverging: red = worse than baseline, blue = better, grey at zero.
DIVERGING = LinearSegmentedColormap.from_list("delta", ["#d03b3b", "#f0efec", "#2a78d6"])

GROUP_TITLE = {"snr_tercile": "By tercile (equal file counts)",
               "snr_band": "By fixed dB edges"}


def save(fig, name):
    fig.savefig(OUT_DIR / name, bbox_inches="tight", dpi=150)
    print(f"saved {OUT_DIR / name}")

In [ ]:
# ── Figure 3 -- F-measure across SNR bins, one line per configuration ─────────
METRIC = "micro_f"          # "macro_f" for the per-file mean instead

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, (by, labels) in zip(axes, GROUPINGS.items()):
    grid = tables[by].reset_index().pivot(index="experiment", columns=by, values=METRIC)
    grid = grid.reindex([k for k in EXPERIMENT_LOGS if k in grid.index])
    x = range(len(labels))
    for label, series in grid.iterrows():
        baseline = label == BASELINE_LABEL
        ax.plot(x, [series.get(l, np.nan) for l in labels],
                marker="o", ms=7, lw=2.6 if baseline else 2,
                color=CONFIG_COLOR[label], zorder=3 if baseline else 2,
                linestyle="-" if baseline else "-", label=label,
                alpha=1.0 if baseline else 0.9)
    counts = tables[by].reset_index().groupby(by, observed=True)["files"].first()
    ax.set_xticks(list(x), [f"{l}\n(n={int(counts.get(l, 0))})" for l in labels])
    ax.set_title(GROUP_TITLE[by], loc="left", fontsize=10)
    ax.set_xlabel("Per-file SNR")
axes[0].set_ylabel(f"{'Micro' if METRIC == 'micro_f' else 'Macro'} F-measure")
axes[1].legend(frameon=False, fontsize=9, loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.suptitle("F-measure against per-file SNR", x=0.09, ha="left", fontsize=11)
fig.tight_layout()
save(fig, "fig_fmeasure_vs_snr_bin.png")
plt.show()

In [ ]:
# ── Figure 4 -- delta vs baseline, config x SNR bin ───────────────────────────
grids = [deltas[(METRIC, by)] for by in GROUPINGS]
vmax = max(np.nanmax(np.abs(g.to_numpy())) for g in grids) or 1.0

fig, axes = plt.subplots(1, 2, figsize=(12, 0.52 * len(grids[0]) + 2.4),
                         gridspec_kw={"width_ratios": [len(g.columns) for g in grids]})
for ax, (by, grid) in zip(axes, zip(GROUPINGS, grids)):
    im = ax.imshow(grid.to_numpy(), cmap=DIVERGING, vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(grid.columns)), grid.columns, fontsize=9)
    ax.set_yticks(range(len(grid.index)), grid.index, fontsize=9)
    ax.set_title(GROUP_TITLE[by], loc="left", fontsize=10)
    ax.grid(False)
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            v = grid.to_numpy()[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:+.1f}", ha="center", va="center", fontsize=9,
                        color="white" if abs(v) > 0.62 * vmax else INK)
axes[1].set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.85, pad=0.02,
             label="Delta F-measure vs Baseline (points)")
fig.suptitle("Where each configuration beats the baseline", x=0.02, ha="left", fontsize=11)
save(fig, "fig_delta_vs_baseline_heatmap.png")
plt.show()

## How far these numbers go

- **Per-file is the finest possible join.** The logs give TP/FP/FN per file, not per event,
  so a per-*event* SNR cannot be tied to individual hits and misses. Doing that would mean
  re-running `src/utils/evaluation.py` with event-level output.
- **The bins hold a handful of files each.** With ~18 scored files, a tercile is ~6 files
  and one noisy recording moves a bin by several F-points. Read ρ and the per-bin deltas as
  indicative; the direction agreeing across *both* groupings and with `snr_med_miss` is
  worth more than any single cell.
- **Each run's threshold is global.** The per-file counts come from the threshold that won
  overall for that run, not the best threshold per SNR bin. A configuration could look weak
  on quiet files simply because its winning threshold was set by the loud ones —
  `threshold_analysis.ipynb` is where that question lives.
- **Micro and macro answer different questions.** Micro is what the challenge reports and
  is dominated by files with many events; macro weights every file equally and is the fairer
  read of "does this help on hard files". Both are in `snr_by_config_bin.csv` — quote which
  one you used.
- **SNR is one axis of difficulty.** Event duration, call density and how much the species
  overlaps other sound all matter too; `event_dur_median_s` rides along in the merged frame
  if you want to check whether it explains more than SNR does.
